Dans ce TP, nous étudions le comptage des mots d’un fichier et comparons quatre approches :
- la méthode séquentielle en Python ;
- une implémentation de MapReduce en Python ;
- Hadoop MapReduce ;
- PySpark avec les DataFrames et les RDD.


# Création de la liste de mots

In [1]:
import random
import time
from collections import defaultdict


In [2]:
liste = ['GRIMACES\n','PULVERISER\n',
'HANTANT\n',
'VILLAGE\n',
'DRESSENT\n',
'DEFERES\n',
'CAPTEES\n',
'PEDANTE\n',
'LIVINGROOM\n',
'BATEAU\n',
'RETIREE\n',
'DISPOSAIT\n',
'SIERA\n',
'ARRETEZ\n',
'REPRENDRONS\n',
'RENAISSANTE\n',
'PAIES\n',
'MONTREALAIS\n',
'PRANGINS\n',
'JARDINER\n',]

In [3]:
with open("mots.txt","w",encoding="utf-8") as f :
    for i in range(1000):
        f.write(random.choice(liste))
fichier = "mots.txt"

# Méthode séquentielle

In [4]:
def compter(fichier):
    with open(fichier,"r",encoding="utf-8") as f :
        mots = f.read().split()
    compteur = {}
    for mot in mots :
        compteur[mot] = compteur.get(mot,0) + 1
    return compteur
start_time = time.perf_counter()
resultat = compter(fichier)
end_time = time.perf_counter()
elapsed_time = end_time - start_time
print(f"Temps d'exécution : {elapsed_time:.6f} secondes")
for mot , nb in sorted (resultat.items(), key= lambda x: x[1], reverse=True):
    print (f"{ mot} : {nb}")

Temps d'exécution : 0.010900 secondes
CAPTEES : 64
PAIES : 63
JARDINER : 55
HANTANT : 54
GRIMACES : 54
MONTREALAIS : 52
DEFERES : 52
DISPOSAIT : 51
DRESSENT : 51
RETIREE : 50
PRANGINS : 49
ARRETEZ : 49
PULVERISER : 48
PEDANTE : 48
LIVINGROOM : 48
SIERA : 46
VILLAGE : 43
BATEAU : 42
RENAISSANTE : 41
REPRENDRONS : 40


On voit bien que cette méthode est extrèmement efficace pour des fichiers d'une centaine de mots voir même d'un millier de mots

Maintenant nous allons passer a la partie suivante ou nous allons répliquer le fonctionnement de MapReduce en Python

# MapReduce en Python

D'abord nous allons mapper donc la nous créons une liste de tuple avec chaque mots de notre fichier associé avec la valeur 1

In [5]:
def mapper(ligne):
    mots = ligne.strip().split()
    return [(mot, 1) for mot in mots]

Ensuite on shuffle. Cela ne reviens pas a tout mélanger mais nous allons regrouper les valeur associés a un même mot

In [6]:
def shuffle(mapped_data):
    grouped = defaultdict(list)
    for mot, valeur in mapped_data:
        grouped[mot].append(valeur)
    return grouped

Pour finir on reduce, c'est a dire qu'on ajoute chaque valeur associé a chaque mots.

In [7]:
def reducer(grouped_data):
    reduced = {}
    for mot, valeurs in grouped_data.items():
        reduced[mot] = sum(valeurs)
    return reduced

Une éxécution ressemblerai ainsi a cela :

In [8]:
mapped = []
start_time = time.perf_counter()
with open (fichier , "r", encoding="utf-8") as f:
    for ligne in f:
        mapped.extend(mapper(ligne))
grouped = shuffle(mapped)
reduced = reducer(grouped)
end_time = time.perf_counter()
elapsed_time = end_time - start_time
print(f"Temps d'exécution : {elapsed_time:.6f} secondes")
for mot , count in sorted (reduced.items(), key= lambda x: x[1], reverse=True):
    print (f"{ mot} : { count }")

Temps d'exécution : 0.001266 secondes
CAPTEES : 64
PAIES : 63
JARDINER : 55
HANTANT : 54
GRIMACES : 54
MONTREALAIS : 52
DEFERES : 52
DISPOSAIT : 51
DRESSENT : 51
RETIREE : 50
PRANGINS : 49
ARRETEZ : 49
PULVERISER : 48
PEDANTE : 48
LIVINGROOM : 48
SIERA : 46
VILLAGE : 43
BATEAU : 42
RENAISSANTE : 41
REPRENDRONS : 40


On voit bien que le compte est exactement le même et que l'ordre de grandeur au niveau du temps d'exécution est aussi le même. On va maintenant passer au MapReduce de Hadoop

# Hadoop MapReduce

Pour la partie de Hadoop nous devrons d'abord avoir une liste bien plus grande de mots car Hadoop n'est pas très performant pour les petits fichiers mais brillent pour les fichier plus volumineux.

In [9]:
with open(fichier,"r",encoding="utf-8") as source:
    contenu = source.read()
fichier_long = "mots_long.txt"
with open(fichier_long,"w",encoding="utf-8") as sortie:
    for _ in range(10000):
        sortie.write(contenu)
        sortie.write("\n")

On va d'abord voir le temps d'exécution avec les méthodes en python :

In [10]:
start_time = time.perf_counter()
resultat_long = compter(fichier_long)
end_time = time.perf_counter()
elapsed_time = end_time - start_time
print(f"Temps d'exécution : {elapsed_time:.6f} secondes")
for mot , nb in sorted (resultat_long.items(), key= lambda x: x[1], reverse=True):
    print (f"{ mot} : {nb}")

Temps d'exécution : 3.030312 secondes
CAPTEES : 640000
PAIES : 630000
JARDINER : 550000
HANTANT : 540000
GRIMACES : 540000
MONTREALAIS : 520000
DEFERES : 520000
DISPOSAIT : 510000
DRESSENT : 510000
RETIREE : 500000
PRANGINS : 490000
ARRETEZ : 490000
PULVERISER : 480000
PEDANTE : 480000
LIVINGROOM : 480000
SIERA : 460000
VILLAGE : 430000
BATEAU : 420000
RENAISSANTE : 410000
REPRENDRONS : 400000


Quand le fichier était seulement 100 fois plus grand cela ne changeait pas grand chose pour Python mais Hadoop mettait 3.866s de calcul maintenant quand le fichier est 10000 fois plus grand Python mets 3.636s et Hadoop met 4,401s. On voit bien que plus la taille du fichier augmente plus cela est long pour Python mais pour Hadoop cela n'augmente pas aussi rapidement.

# PySpark

PySpark permet également de compter les occurrences des mots. Nous commençons par l’API DataFrame, puis nous réalisons le même WordCount avec les RDD.


**Prérequis :** utiliser l’environnement Conda `spark311` (Python 3.11) comme kernel Jupyter et disposer de Java 17 dans `C:\Java\jdk-17`. Après un changement d’environnement, redémarrer le kernel avant de lancer cette cellule. Spark sera exécuté en mode local, sans dépendre de YARN.

In [11]:
import os
import sys
from pathlib import Path

# Configurer Java et Python AVANT la première création de SparkSession.
if sys.version_info < (3, 10):
    raise RuntimeError("Utiliser le kernel Python 3.11 de l'environnement spark311.")

java_home = Path(r"C:\Java\jdk-17")
if not (java_home / "bin" / "java.exe").is_file():
    raise FileNotFoundError(f"Java 17 introuvable dans {java_home}")

os.environ["JAVA_HOME"] = str(java_home)
os.environ["PATH"] = str(java_home / "bin") + os.pathsep + os.environ["PATH"]
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"
os.environ["SPARK_LOCAL_HOSTNAME"] = "localhost"

from pyspark.sql import SparkSession, functions as F

spark = (
    SparkSession.builder
    .appName("TP1_WordCount")
    .master("local[2]")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.pyspark.python", sys.executable)
    .getOrCreate()
)

print("Python :", sys.version.split()[0])
print("Java Spark :", spark.sparkContext._jvm.java.lang.System.getProperty("java.version"))
print("Spark :", spark.version)

Python : 3.11.17
Java Spark : 17.0.12
Spark : 4.0.4


In [12]:
# Lire le même fichier que pour les autres méthodes.
chemin = Path(fichier_long).resolve().as_uri()

debut = time.perf_counter()
df = spark.read.text(chemin)

resultat_df = (
    df.select(F.explode(F.split(F.col("value"), r"\s+")).alias("mot"))
      .filter(F.col("mot") != "")
      .groupBy("mot")
      .count()
      .orderBy(F.desc("count"), F.asc("mot"))
)

# show() déclenche effectivement le calcul.
resultat_df.show(20, truncate=False)
fin = time.perf_counter()
print(f"Temps d'exécution DataFrame : {fin - debut:.3f} secondes")

+-----------+------+
|mot        |count |
+-----------+------+
|CAPTEES    |640000|
|PAIES      |630000|
|JARDINER   |550000|
|GRIMACES   |540000|
|HANTANT    |540000|
|DEFERES    |520000|
|MONTREALAIS|520000|
|DISPOSAIT  |510000|
|DRESSENT   |510000|
|RETIREE    |500000|
|ARRETEZ    |490000|
|PRANGINS   |490000|
|LIVINGROOM |480000|
|PEDANTE    |480000|
|PULVERISER |480000|
|SIERA      |460000|
|VILLAGE    |430000|
|BATEAU     |420000|
|RENAISSANTE|410000|
|REPRENDRONS|400000|
+-----------+------+

Temps d'exécution DataFrame : 9.053 secondes


## WordCount avec les RDD PySpark

On vérifie d’abord que les workers Python fonctionnent, puis on applique `flatMap`, `map` et `reduceByKey` au même fichier `mots_long.txt`. La méthode `takeOrdered` déclenche le calcul et récupère les 20 premiers résultats.

In [13]:
# Test rapide du fonctionnement des RDD Python.
test_rdd = spark.sparkContext.parallelize([1, 2, 3], 1)
print("Test RDD :", test_rdd.map(lambda x: x * 2).collect())  # [2, 4, 6]

Test RDD : [2, 4, 6]


In [14]:
debut = time.perf_counter()

rdd = spark.sparkContext.textFile(chemin)
resultat_rdd = (
    rdd.flatMap(lambda ligne: ligne.split())
       .map(lambda mot: (mot, 1))
       .reduceByKey(lambda a, b: a + b)
)

top20_rdd = resultat_rdd.takeOrdered(20, key=lambda x: (-x[1], x[0]))
fin = time.perf_counter()

for mot, nb in top20_rdd:
    print(f"{mot} : {nb}")

print(f"Temps d'exécution RDD : {fin - debut:.3f} secondes")

CAPTEES : 640000
PAIES : 630000
JARDINER : 550000
GRIMACES : 540000
HANTANT : 540000
DEFERES : 520000
MONTREALAIS : 520000
DISPOSAIT : 510000
DRESSENT : 510000
RETIREE : 500000
ARRETEZ : 490000
PRANGINS : 490000
LIVINGROOM : 480000
PEDANTE : 480000
PULVERISER : 480000
SIERA : 460000
VILLAGE : 430000
BATEAU : 420000
RENAISSANTE : 410000
REPRENDRONS : 400000
Temps d'exécution RDD : 24.084 secondes


# Conclusion
On ne peut directement dire que le RDD est moins efficace en terme de calcul. En effet, dans ce cas Spark fait des échanges avec java et cela augmente le temps d'exécution. Mais les DataFrames de PySpark sont quand même optimisés pour ce genre de calcul.